In [1]:
import numpy as np

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import CXGate, PauliEvolutionGate, QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration

from qiskit.transpiler import PassManager, Layout
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy, FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router_rzz import CommutingGateRouterRzz
from qiskit_qaoa.utils.commuting_gate_router import CommutingGateRouter
from qiskit_qaoa.hubo.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from qiskit_qaoa.utils.gfa_utils import gfa_file_to_graph
from qiskit_qaoa.utils.sat_mapper import HigherOrderSatMapper
from qiskit_qaoa.utils.hamiltonian_utils import hamiltonian_to_interactions



In [2]:
def print_circuit_info(qc: QuantumCircuit, circuit_name):
    print(f'{circuit_name} has {qc.num_qubits} qubits')
    print(f'{circuit_name} has {qc.num_nonlocal_gates()} non-local gates and {qc.depth(lambda instr: len(instr.qubits) > 1)} non-local depth')
    print(f'{circuit_name} contains {list(qc.count_ops().keys())} gates.')

In [3]:
method = 'statevector'
backend_options = dict(
    method=method,
    device='CPU',
    precision='single',
    basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx"]
)


In [4]:
results = {}
mapper = HigherOrderSatMapper(timeout=30)

for filename, copy_numbers in zip(
    [
        'test_N2_W2', 'trivial', 'test_N3_W4', 'test_N4_W5', 'test_N4_W6', 
        # 'test_N5_W6', 'test_N7_W2', 'test_N7_W3','test_N7_W4', 'test_N7_W5', 
        # 'test_N8_W2', 'test_N8_W3','test_N8_W4', 'test_N8_W5', 'test_N8_W6',
        # 'test_N9_W6', 'test_N10_W6','test_N14_W7'
    ], 
    [
        [1,1], [1,1,1], [2,1,1], [2,1,1,1], [2,2,1,1],
        # [1,2,1,1,1], [1,0,0,0,0,0,1], [1,1,0,0,0,0,1], [1,1,1,0,0,0,1], [1,1,1,0,1,0,1],
        # [1,0,0,0,0,0,0,1],[1,1,0,0,0,0,0,1],[1,1,1,0,0,0,0,1],[1,1,1,1,0,0,0,1],[1,1,0,1,1,1,0,1],
        # [1,1,0,0,1,0,1,1,1], [1,1,0,0,1,0,1,1,0,1], [1,1,0,0,1,0,1,0,0,1,0,0,1,1]
    ]
):
# for filename, copy_numbers in zip(
#     [
#         'test_N3_W4'
#     ], 
#     [
#         [2,1,1]
#     ]
# ):

    filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
    graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)
    hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
    ess = ExtendedSwapStrategy.from_grid(n, T)
    num_physical_qubits = ess._num_vertices

    program_interactions = hamiltonian_to_interactions(hamiltonian, 0.0, 1.0)

    config = AerSimulator._DEFAULT_CONFIGURATION
    config["n_qubits"] = num_physical_qubits
    config = AerBackendConfiguration.from_dict(config)
    backend = AerSimulator(configuration=config, coupling_map=ess._coupling_map, **backend_options)
    backend.set_option("n_qubits", num_physical_qubits)

    default_qaoa = QAOAAnsatz(hamiltonian, reps=1, initial_state= QuantumCircuit(num_physical_qubits), mixer_operator=QuantumCircuit(num_physical_qubits))
    t_default_qaoa = transpile(default_qaoa, backend=backend, optimization_level=3,basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx"])


    best_rzz_depth = np.inf
    best_rzz_count = np.inf
    best_rzz_layers = 0
    best_rz_depth = np.inf
    best_rz_count = np.inf
    best_rz_layers = 0

    # layers = sorted(list(set([int(x) for x in np.linspace(0, len(ess._swap_layers), 10)])))
    layers = [10]
    for layer in layers:
        pm = PassManager(
            [
                FindCommutingPauliEvolutionsMulti(), 
                CommutingGateRouterRzz(
                    ess,
                    max_layers=layer,
                    perform_extra_swaps=True
                ),
                SwapToFinalMapping(),
                InverseCancellation(gates_to_cancel=[CXGate()]),
                CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
                InverseCancellation(gates_to_cancel=[CXGate()]),
            ]
        )
        pm_rz = PassManager(
            [
                FindCommutingPauliEvolutionsMulti(), 
                CommutingGateRouter(
                    ess,
                    max_layers=layer,
                    perform_extra_swaps=True
                ),
                SwapToFinalMapping(),
                InverseCancellation(gates_to_cancel=[CXGate()]),
                CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
                InverseCancellation(gates_to_cancel=[CXGate()]),
            ]
        )


        sat_results = mapper.hubo_max_sat(
            num_physical_qubits, program_interactions, ess, layer
        )
        if sat_results is None:
            print('No results')
            continue
        mapping = sat_results[layer][1]
        edge_map = dict(mapping)
        donor_qc = QuantumCircuit(num_physical_qubits)
        layout = Layout({donor_qc.qubits[key]: val for key, val in edge_map.items()})

        qc = QuantumCircuit(num_physical_qubits)
        qc.append(PauliEvolutionGate(hamiltonian), range(num_physical_qubits))
        tqc = pm.run(qc)
        tqc_rz = pm_rz.run(qc)

        rzz_depth = tqc.depth(lambda instr: len(instr.qubits) > 1)
        if rzz_depth < best_rzz_depth:
            best_rzz_depth = rzz_depth
            best_rzz_count = tqc.num_nonlocal_gates()
            best_rzz_layers = layer

        rz_depth = tqc_rz.depth(lambda instr: len(instr.qubits) > 1)
        if rz_depth < best_rz_depth:
            best_rz_depth = rz_depth
            best_rz_count = tqc_rz.num_nonlocal_gates()
            best_rz_layers = layer
    
    results[filename] = {
        'default': (t_default_qaoa.num_nonlocal_gates(), t_default_qaoa.depth(lambda instr: len(instr.qubits) > 1)),
        'rz': (best_rz_count, best_rz_depth, best_rz_layers),
        'rzz': (best_rzz_count, best_rzz_depth, best_rzz_layers),
    }
        


Keeping constraints at times: [0]


/nfs/users/nfs_j/jc59/quantumwork/pangenome/.venv/lib/python3.10/site-packages/qiskit/compiler/transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


09:19:01 - qiskit_qaoa.utils.sat_mapper - INFO - Num layers: 10
09:19:01 - qiskit_qaoa.utils.sat_mapper - INFO - Re-computing distance tensor
09:19:01 - qiskit_qaoa.utils.sat_mapper - INFO - Re-computing distance tensor
09:19:01 - qiskit_qaoa.utils.swap_strategy - INFO - Loaded data
09:19:01 - qiskit_qaoa.utils.sat_mapper - INFO - Re-computing distance tensor
09:19:01 - qiskit_qaoa.utils.swap_strategy - INFO - Loaded data
09:19:01 - qiskit_qaoa.utils.sat_mapper - INFO - Hard constraints: 186
09:19:01 - qiskit_qaoa.utils.sat_mapper - INFO - Soft constraints: 1986
Max layers needed to apply swap decompose: 3
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Max layers needed to apply swap decompose: 3
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Keeping constraints at times: [0 1]
09:19:02 - qiskit_qaoa.utils.sat_mapper - INFO - Num layers: 10
09:19:02 - qiskit_qaoa.utils.sat_mapper - INFO - Re-computing distance tensor
09:19:02 - qiskit_qaoa.utils.sat_

/nfs/users/nfs_j/jc59/quantumwork/pangenome/.venv/lib/python3.10/site-packages/qiskit/compiler/transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


09:19:02 - qiskit_qaoa.utils.sat_mapper - INFO - Hard constraints: 657
09:19:02 - qiskit_qaoa.utils.sat_mapper - INFO - Soft constraints: 23778
Max layers needed to apply swap decompose: 6
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Max layers needed to apply swap decompose: 6
Gates we cannot directly implement: 0
[]
Transpiling accumulator
Keeping constraints at times: [1 2 0]
09:19:02 - qiskit_qaoa.utils.sat_mapper - INFO - Num layers: 10
09:19:02 - qiskit_qaoa.utils.sat_mapper - INFO - Re-computing distance tensor
09:19:02 - qiskit_qaoa.utils.sat_mapper - INFO - Re-computing distance tensor
09:19:02 - qiskit_qaoa.utils.swap_strategy - INFO - Loaded data
09:19:02 - qiskit_qaoa.utils.sat_mapper - INFO - Re-computing distance tensor
09:19:02 - qiskit_qaoa.utils.swap_strategy - INFO - Loaded data
09:19:03 - qiskit_qaoa.utils.sat_mapper - INFO - Hard constraints: 1596
09:19:03 - qiskit_qaoa.utils.sat_mapper - INFO - Soft constraints: 60060
Max layers needed to apply 

...

09:20:31 - qiskit_qaoa.utils.sat_mapper - INFO - Hard constraints: 13272


INFO:qiskit_qaoa.utils.sat_mapper:Hard constraints: 13272


09:20:31 - qiskit_qaoa.utils.sat_mapper - INFO - Soft constraints: 4309704


INFO:qiskit_qaoa.utils.sat_mapper:Soft constraints: 4309704


Max layers needed to apply swap decompose: 10
Gates we cannot directly implement: 415
[(8, 9, 12, 14, 15), (8, 10, 13, 14, 15), (13, 15, 16, 18), (8, 9, 10, 11, 12, 14), (16, 17, 18, 19, 20, 23), (8, 9, 11, 14, 15), (0, 16), (1, 2, 4, 5, 6), (12, 14, 17, 18, 19), (9, 10, 11, 12, 14), (5, 6, 8, 9, 10), (12, 13, 21), (10, 13, 14), (14, 16, 17, 18), (8, 9, 10, 16, 17, 18), (3, 4, 6), (12, 15, 16, 17, 19), (10, 11, 13), (12, 15, 16, 18, 19), (17, 18, 19, 21), (10, 11, 14), (12, 14, 22), (4, 7, 9, 10), (13, 16, 18, 19), (12, 13, 14, 17, 18), (0, 1, 12, 13), (0, 2, 5, 6), (10, 11, 13, 14), (4, 5, 6, 13, 14), (16, 17, 19, 20, 23), (8, 10, 11, 14), (1, 3, 4, 5, 7), (4, 6, 10), (0, 2, 16, 18), (16, 19, 22), (4, 5, 6, 9, 10), (8, 9, 10, 11, 12, 15), (8, 11, 12, 13, 14, 15), (9, 11, 12, 14, 15), (16, 18, 19, 21, 22), (10, 11, 12, 13), (14, 16, 18), (5, 6, 7, 9, 10), (16, 17, 19, 23), (1, 12, 13), (4, 5, 8, 10), (4, 5, 6, 9, 10, 11), (4, 6, 18), (0, 3, 4, 5, 6), (0, 1, 2, 21, 22), (3, 4, 6, 7), (1

{
8: 0,
4: 1,
0: 2,
9: 3,
6: 4,
1: 5,
5: 6,
10: 7,
2: 8,
3: 9,
7: 10,
11: 11
}

In [ ]:
results

TODO: error in N9_W6 for Rz router only?
Looks like unsuitable subset chosen

In [ ]:
# mapper = HigherOrderSatMapper(timeout=30)

# filename, copy_numbers = 'test_N9_W6' , [1,1,0,0,1,0,1,1,1]
    
# filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
# graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)
# hamiltonian = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
# ess = ExtendedSwapStrategy.from_grid(n, T)
# num_physical_qubits = ess._num_vertices

# program_interactions = hamiltonian_to_interactions(hamiltonian, 0.0, 1.0)

# config = AerSimulator._DEFAULT_CONFIGURATION
# config["n_qubits"] = num_physical_qubits
# config = AerBackendConfiguration.from_dict(config)
# backend = AerSimulator(configuration=config, coupling_map=ess._coupling_map, **backend_options)
# backend.set_option("n_qubits", num_physical_qubits)

# default_qaoa = QAOAAnsatz(hamiltonian, reps=1, initial_state= QuantumCircuit(num_physical_qubits), mixer_operator=QuantumCircuit(num_physical_qubits))
# t_default_qaoa = transpile(default_qaoa, backend=backend, optimization_level=3,basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx"])


# layer = 42
# pm = PassManager(
#     [
#         FindCommutingPauliEvolutionsMulti(), 
#         CommutingGateRouterRzz(
#             ess,
#             max_layers=layer,
#             perform_extra_swaps=True
#         ),
#         SwapToFinalMapping(),
#         InverseCancellation(gates_to_cancel=[CXGate()]),
#         CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
#         InverseCancellation(gates_to_cancel=[CXGate()]),
#     ]
# )
# pm_rz = PassManager(
#     [
#         FindCommutingPauliEvolutionsMulti(), 
#         CommutingGateRouter(
#             ess,
#             max_layers=layer,
#             perform_extra_swaps=True
#         ),
#         SwapToFinalMapping(),
#         InverseCancellation(gates_to_cancel=[CXGate()]),
#         CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
#         InverseCancellation(gates_to_cancel=[CXGate()]),
#     ]
# )


# donor_qc = QuantumCircuit(num_physical_qubits)


# sat_results = mapper.hubo_max_sat(
#     num_physical_qubits, program_interactions, ess, layer
# )
# if sat_results is None:
#     raise Exception('No results')
    
    
# mapping = sat_results[layer][1]
# edge_map = dict(mapping)
# layout = Layout({donor_qc.qubits[key]: val for key, val in edge_map.items()})

# # Phys: Virt
# # layout_dict = {4: 0, 8: 1, 3: 2, 7: 3, 9: 4, 5: 5, 1: 6, 6: 7, 2: 8, 0: 9}
# # layout = Layout({key: donor_qc.qubits[val] for key, val in layout_dict.items()})

# qc = QuantumCircuit(num_physical_qubits)
# qc.append(PauliEvolutionGate(hamiltonian), range(num_physical_qubits))
# tqc = pm.run(qc)
# tqc_rz = pm_rz.run(qc)

# rzz_depth = tqc.depth(lambda instr: len(instr.qubits) > 1)
# rz_depth = tqc_rz.depth(lambda instr: len(instr.qubits) > 1)

In [ ]:
ess = ExtendedSwapStrategy.from_grid(3, 4)
router = CommutingGateRouterRzz(
    ess,
    max_layers=0,
    perform_extra_swaps=True
)

In [ ]:
router._compute_interaction(
    (3, 4, 5, 6, 7),
    (4,5),
    PauliEvolutionGate(SparsePauliOp(['I']*12)),
    QuantumCircuit(12),
    {x: set([x]) for x in range(12)}
)

In [ ]:
router._suitable_superset(
    (3, 4, 5, 6, 7), 
    None, 
    [(0, 2, 4, 5, 7, 9)], 
    {0: {0}, 1: {0, 1}, 2: {2}, 3: {0, 1, 3}, 4: {2, 4}, 5: {0, 1, 3, 5, 7, 9}, 6: {6}, 7: {9, 7}, 8: {8}, 9: {9}}
)

In [ ]:
router._missing_info_is_connected(set([1,3]),(4,5), {0: {0}, 1: {0, 1}, 2: {2}, 3: {0, 1, 3}, 4: {2, 4}, 5: {0, 1, 3, 5, 7, 9}, 6: {6}, 7: {9, 7}, 8: {8}, 9: {9}})

In [ ]:
ess = ExtendedSwapStrategy.from_grid(9, 6)
rz_router = CommutingGateRouter(
    ess,
    max_layers=42,
    perform_extra_swaps=True
)

In [ ]:
rz_router._suitable_subset(
    (0, 1, 2, 3, 4, 6, 7, 8),
    1,
    [(1,2,3,4)],
    {0: {0, 6, 7}, 1: {0, 1, 2, 3, 4, 6, 7, 8}, 2: {2, 3, 4}, 3: {3, 4}, 4: {4}, 5: {5}, 6: {6, 7}, 7: {8, 7}, 8: {8}, 9: {9}, 10: {10}, 11: {11}, 12: {12}, 13: {13}, 14: {14}, 15: {15}, 16: {16}, 17: {17}, 18: {18}, 19: {19}, 20: {20}, 21: {21}, 22: {22}, 23: {23}, 24: {24}, 25: {25}, 26: {26}, 27: {27}, 28: {28}, 29: {29}}
)

In [ ]:
print_circuit_info(tqc, 'rzz')

In [ ]:
print_circuit_info(tqc_rz, 'rz')

In [ ]:
tqc.draw(fold=-1)

In [ ]:
tqc_rz.draw(fold=-1)